# Citation Index API — Usage Guide

This notebook demonstrates how to use the **Citation Index API** to extract and parse academic references from PDF documents.

The API exposes three main pipelines, all backed by an asynchronous job queue:

| Pipeline | Endpoint | Input | Output |
|----------|----------|-------|--------|
| **Text Extraction** | `POST /extract/text` | PDF file | Markdown text |
| **Reference Extraction** | `POST /extract/references` | Markdown text | List of raw reference strings |
| **Reference Parsing** | `POST /parse/references` | List of reference strings | Structured bibliographic records |

Every endpoint returns a `job_id` immediately. You then poll for status and retrieve the result once the job completes.

## 0. Setup

In [1]:
import time
import json
import requests
from pathlib import Path
from pprint import pprint

In [2]:
# Point this at your running Citation Index API instance
API_BASE = "https://citation-index-api-graphia-app1-staging.apps.bst2.paas.psnc.pl"

# Polling settings
POLL_INTERVAL = 3   # seconds between status checks
MAX_WAIT     = 600  # maximum seconds to wait for a job

## 1. Health Check

Verify the API is reachable and its backing services (Redis, storage) are healthy.

In [3]:
resp = requests.get(f"{API_BASE}/health")
resp.raise_for_status()
pprint(resp.json())

{'redis': 'ok', 'status': 'healthy', 'storage': 'ok', 'version': '0.1.0'}


---

## 2. Helper: Poll a Job Until Completion

All three pipelines are **asynchronous**. The helper below submits a polling loop that:

1. Calls `GET /jobs/{job_id}/status` every few seconds.
2. Returns the final result from `GET /jobs/{job_id}` once the status is `completed`.
3. Raises on failure or timeout.

In [4]:
def wait_for_job(job_id: str, poll_interval: int = POLL_INTERVAL, max_wait: int = MAX_WAIT) -> dict:
    """Poll a job until it reaches a terminal state and return the result."""
    start = time.time()
    last_status = None

    while (time.time() - start) < max_wait:
        status_resp = requests.get(f"{API_BASE}/jobs/{job_id}/status")
        status_resp.raise_for_status()
        info = status_resp.json()
        status = info["status"]

        if status != last_status:
            elapsed = time.time() - start
            print(f"  [{elapsed:5.1f}s] status = {status}")
            last_status = status

        if status == "completed":
            result_resp = requests.get(f"{API_BASE}/jobs/{job_id}")
            result_resp.raise_for_status()
            return result_resp.json()

        if status == "failed":
            raise RuntimeError(f"Job {job_id} failed: {info.get('error', 'unknown')}")

        time.sleep(poll_interval)

    raise TimeoutError(f"Job {job_id} did not complete within {max_wait}s")

---

## 3. Text Extraction (PDF → Markdown)

Upload a PDF and get back markdown-formatted text.

**Endpoint:** `POST /extract/text`  
**Query params:**
- `extractor` — `"pymupdf"` (default) or `"marker"`
- `markdown` — `true` (default) / `false`

**Body:** multipart file upload (`file` field, `application/pdf`)

In [5]:
# --- Replace with the path to your own PDF ---
pdf_path = Path("../benchmarks/cex/all_pdfs/COM-SCI_25.pdf")

with open(pdf_path, "rb") as f:
    resp = requests.post(
        f"{API_BASE}/extract/text",
        files={"file": (pdf_path.name, f, "application/pdf")},
        params={"extractor": "pymupdf", "markdown": True},
    )
resp.raise_for_status()

job = resp.json()
print(f"Job submitted: {job['job_id']}  (status: {job['status']})")

Job submitted: 68d334fa-17d2-499f-bfec-5bcdeeedf0c0  (status: queued)


In [6]:
text_result = wait_for_job(job["job_id"])

extracted_text = text_result["text"]
print(f"Extracted {len(extracted_text)} characters.")
print("\n--- First 500 characters ---")
print(extracted_text[:500])

  [  0.1s] status = processing
  [ 12.6s] status = completed
Extracted 83801 characters.

--- First 500 characters ---
J Real-Time Image Proc (2019) 16:1607–1628

DOI 10.1007/s11554-017-0669-4


ORIGINAL RESEARCH PAPER

# Analog signal processing solution for machine vision applications


Nihar Athreyas [1] [•] Dev Gupta [1] [•] Jai Gupta [2]


Received: 18 June 2016 / Accepted: 17 January 2017 / Published online: 6 February 2017

- Springer-Verlag Berlin Heidelberg 2017



Abstract The field of machine vision is continuously
evolving. There are new products coming into the market
that have very severe size, wei


---

## 4. Reference Extraction (Text → Reference Strings)

Send markdown text and get back a list of raw bibliographic reference strings identified by the LLM.

**Endpoint:** `POST /extract/references`  
**Query params:**
- `method` — `"full_text"` (default)
- `temperature` — float, e.g. `0.3`
- `prompt_name` — custom prompt template path (optional)

**Body (JSON):** `{"text": "<markdown text>"}`

In [7]:
# You can use the text extracted in section 3, or supply your own.
sample_text = """
## Literatur

Aron, Raymond/Dominique Schnapper (1988): Power, modernity, and sociology:
selected sociological writings. Aldershot, Hants, England

Collins, Harry (2004): Gravity's shadow: the search for gravitational waves.
Chicago: University of Chicago Press.

Collins, Harry M. (1981): Stages in the Empirical Programme of Relativism.
In: Social Studies of Science, 11 S. 3-10.

Dosi, Giovanni (1982): Technological Paradigms and Technological Trajectories.
In: Research Policy, 11 S. 147-162.

Kuhn, Thomas S. (1976): Die Struktur wissenschaftlicher Revolutionen.
Frankfurt/Main: Suhrkamp.
"""

resp = requests.post(
    f"{API_BASE}/extract/references",
    json={"text": sample_text},
    params={"method": "full_text", "temperature": 0.3},
)
resp.raise_for_status()

job = resp.json()
print(f"Job submitted: {job['job_id']}  (status: {job['status']})")

Job submitted: acde8975-381b-4b3a-9200-e6e2cfe9547f  (status: queued)


In [8]:
extraction_result = wait_for_job(job["job_id"])

references = extraction_result.get("references", [])
print(f"Found {len(references)} references:\n")
for i, ref in enumerate(references, 1):
    print(f"  {i}. {ref}")

  [  0.1s] status = processing
  [  3.2s] status = completed
Found 5 references:

  1. Aron, Raymond/Dominique Schnapper (1988): Power, modernity, and sociology: selected sociological writings. Aldershot, Hants, England
  2. Collins, Harry (2004): Gravity's shadow: the search for gravitational waves. Chicago: University of Chicago Press.
  3. Collins, Harry M. (1981): Stages in the Empirical Programme of Relativism. In: Social Studies of Science, 11 S. 3-10.
  4. Dosi, Giovanni (1982): Technological Paradigms and Technological Trajectories. In: Research Policy, 11 S. 147-162.
  5. Kuhn, Thomas S. (1976): Die Struktur wissenschaftlicher Revolutionen. Frankfurt/Main: Suhrkamp.


---

## 5. Reference Parsing (Strings → Structured Records)

Send a list of raw reference strings and receive structured bibliographic fields (author, title, year, journal, etc.).

**Endpoint:** `POST /parse/references`  
**Query params:**
- `parser` — `"llm"` (default) or `"grobid"`
- `temperature` — float, e.g. `0.0`
- `prompt_name` — custom prompt template path (optional)

**Body (JSON):** `{"references": ["ref string 1", "ref string 2", ...]}`

In [ ]:
# You can feed in the references from section 4, or supply your own list.
raw_references = [
    "B Algers, G Bertoni, D Broom, J Hartung, L Lidfors, J Metz, L Munksgaard, "
    "T N Pina, P Oltenacu, J Rehage, J Rushen. Scientific report on the effects "
    "of farming systems on dairy cow welfare and disease. Annex to the EFSA Journal. "
    "2009. Vol. 1143",
    "E Burow, T Rousing, P Thomsen, D Otten, J Sørensen. Effect of grazing on "
    "the cow welfare of dairy herds evaluated by a multidimensional welfare index. "
    "Animal. 2013a. Vol. 7",
    "D Gieseke, C Lambertz, M Gauly. Relationship between herd size and animal "
    "welfare in dairy cattle. Journal of Dairy Science. 2018. Vol. 101",
    "S. Superman, B. Batman, C. Catwoman, and S. Spiderman. 2000. Superheroes experiences with books, 20th edition. The Phantom Editors Associates, Gotham City."
]

resp = requests.post(
    f"{API_BASE}/parse/references",
    json={"references": raw_references},
    params={"parser": "llm", "temperature": 0.0},
)
resp.raise_for_status()

job = resp.json()
print(f"Job submitted: {job['job_id']}  (status: {job['status']})")

Job submitted: a3b15103-37ce-458c-a22c-41dafa544b2b  (status: queued)


In [10]:
parse_result = wait_for_job(job["job_id"])

parsed_refs = parse_result.get("references", [])
print(f"Parsed {len(parsed_refs)} references:\n")
for ref in parsed_refs:
    pprint(ref)
    print()

  [  0.1s] status = processing
  [  3.2s] status = completed
Parsed 3 references:

{'authors': [{'first_name': 'B',
              'middle_name': ',',
              'name_link': None,
              'role_name': None,
              'surname': 'Algers'},
             {'first_name': 'G',
              'middle_name': ',',
              'name_link': None,
              'role_name': None,
              'surname': 'Bertoni'},
             {'first_name': 'D',
              'middle_name': ',',
              'name_link': None,
              'role_name': None,
              'surname': 'Broom'},
             {'first_name': 'J',
              'middle_name': ',',
              'name_link': None,
              'role_name': None,
              'surname': 'Hartung'},
             {'first_name': 'L',
              'middle_name': ',',
              'name_link': None,
              'role_name': None,
              'surname': 'Lidfors'},
             {'first_name': 'J',
              'middle_name': ',',
   

---

## 6. End-to-End: PDF → Structured References

## WIP 

---

## 7. Checking Job Status Directly

You can inspect any job at any time using its `job_id`.

In [12]:
# Replace with a real job_id from a previous run
example_job_id = job['job_id']  # reuse the last one from above

# GET /jobs/{job_id}/status — lightweight status check
status_resp = requests.get(f"{API_BASE}/jobs/{example_job_id}/status")
print("Status endpoint:")
pprint(status_resp.json())

Status endpoint:
{'completed_at': '2026-02-19T08:43:50.876874',
 'completed_stages': ['reference_parsing'],
 'created_at': '2026-02-19T08:43:49.467034',
 'current_stage': 'reference_parsing',
 'error': None,
 'job_id': 'a3b15103-37ce-458c-a22c-41dafa544b2b',
 'progress': None,
 'started_at': None,
 'status': 'completed'}


In [13]:
# GET /jobs/{job_id} — full result (only meaningful when status = completed)
result_resp = requests.get(f"{API_BASE}/jobs/{example_job_id}")
print(f"Result endpoint (HTTP {result_resp.status_code}):")
pprint(result_resp.json())

Result endpoint (HTTP 200):
{'count': 3,
 'parser': 'llm',
 'references': [{'authors': [{'first_name': 'B',
                              'middle_name': ',',
                              'name_link': None,
                              'role_name': None,
                              'surname': 'Algers'},
                             {'first_name': 'G',
                              'middle_name': ',',
                              'name_link': None,
                              'role_name': None,
                              'surname': 'Bertoni'},
                             {'first_name': 'D',
                              'middle_name': ',',
                              'name_link': None,
                              'role_name': None,
                              'surname': 'Broom'},
                             {'first_name': 'J',
                              'middle_name': ',',
                              'name_link': None,
                              'role_name': Non

You can also request XML output for parsing results (**WIP**)

```
GET /jobs/{job_id}?format=xml
```

In [ ]:
xml_resp = requests.get(f"{API_BASE}/jobs/{example_job_id}", params={"format": "xml"})
if xml_resp.status_code == 200:
    print(xml_resp.text[:1000])
else:
    print(f"XML not available (HTTP {xml_resp.status_code}): {xml_resp.text}")

---

##  Batch Processing with Concurrent Requests

The queue system supports multiple concurrent jobs. Submit several at once, then poll them in parallel.

Current status: max 3 parallel jobs for none LLM tasks (text extraction/grobid), max 6 for LLM tasks (reference extraction/parsing)







---

## API Reference (Quick Cheat Sheet)

| Method | Path | Description |
|--------|------|-------------|
| `GET`  | `/` | API info (name, version, links) |
| `GET`  | `/health` | Health check (Redis + storage status) |
| `GET`  | `/jobs/{job_id}/status` | Lightweight job status |
| `GET`  | `/jobs/{job_id}` | Full job result (`?format=xml` for XML) |
| `POST` | `/extract/text` | Upload PDF → markdown text |
| `POST` | `/extract/references` | Markdown text → reference strings |
| `POST` | `/parse/references` | Reference strings → structured records |

Interactive Swagger docs are available at **`{API_BASE}/docs`**.